# Data 552 - Group 4 Project: Data Driven Planning for Fast-Charging Network Expansion in Canada for Electric Vehicles

# EV Charging Station Dataset — Data Wrangling & Cleaning Report

## Overview

The dataset sourced from Kaggle contains **1,719 records** of electric vehicle (EV) charging stations across Canada, originally spanning **39 columns**. Notably, the dataset is dominated by **DC Fast (Level 3) chargers** — `ev_dc_fast_count` is 100% populated with a mean of 2.53 ports per station, while `ev_level2_evse_num` is 83% null, directly reflecting that the majority of stations are Level 3 only. The goal of wrangling was twofold:

1. Prepare a clean dataset for **geographic distribution analysis** (plotting station locations on a Canadian map)
2. Prepare a fully numeric dataset for **machine learning** to predict future demand and optimal installation locations

The pipeline reduced the dataset to **28 columns** after cleaning, then split it into two purpose-built output files.

---

## Step 1 — Dropping Useless Columns

**39 → 28 columns**

Ten columns were removed for the following reasons:

| Column | Reason for Removal |
|---|---|
| `Fuel Type Code` | Zero variance — every row was `ELEC` |
| `Country` | Zero variance — every row was `CA` |
| `Plus4` | 100% missing |
| `Hydrogen Status Link` | 100% missing |
| `EV Other Info` | 100% missing |
| `BD Blends (French)` | 100% missing |
| `EV On-Site Renewable Source` | ~100% missing |
| `Intersection Directions` | 99% missing |
| `Access Days Time (French)` | Duplicate of English column |
| `Groups With Access Code (French)` | Duplicate of English column |
| `EV Pricing (French)` | Duplicate of English column |

---

## Step 2 — Column Renaming

All column names were standardised to **snake_case** to avoid spacing issues and improve code readability.

**Example:** `EV DC Fast Count` → `ev_dc_fast_count`

---

## Step 3 — Fixing Data Types

Columns were cast to their correct types to enable proper computation:

| Type | Columns |
|---|---|
| `datetime64` | `open_date`, `date_last_confirmed`, `expected_date`, `updated_at` |
| `float` / `int` | `latitude`, `longitude`, `ev_dc_fast_count`, `ev_level2_evse_num`, `id` |
| Binary `0`/`1` | `restricted_access` |

---

## Step 4 — Handling Missing Values

Rather than dropping rows with missing data (which would reduce the dataset), missing values were imputed based on the nature of each column:

| Column | Missing % | Strategy | Rationale |
|---|---|---|---|
| `ev_level2_evse_num` | 83% | Filled with `0` | The dataset is dominated by DC Fast (Level 3) chargers. The 83% null rate reflects that these stations have no Level 2 ports — null means `0`, not unknown |
| `station_phone` | 16% | Filled with `"Unknown"` | Not critical for analysis |
| `owner_type_code` | 44% | Filled with `"Unknown"` | Preserves rows for ML |
| `facility_type` | 64% | Filled with `"Unknown"` | Preserves rows for ML |
| `ev_pricing` | 77% | Filled with `"Unknown"` | Preserves rows for ML |
| `restricted_access` | 52% | Filled with `0` | Assumed not restricted |
| `ev_connector_types` | 10% | Filled with `"Unknown"` | Preserves rows for ML |

All 1,719 rows were retained after imputation.

---

## Step 5 — Geographic Cleaning

Two geographic checks were applied:

- **Coordinate validation:** Rows with missing or out-of-Canada coordinates (latitude: 41–84°N, longitude: 141–52°W) were flagged for removal. No rows required removal in this dataset.
- **Province name mapping:** Province codes were expanded to full names for readable map labels (e.g. `QC` → `Quebec`, `ON` → `Ontario`).

---

## Step 6 — Encoding Categorical Variables

Categorical columns were encoded into numeric form for machine learning compatibility:

**One-Hot Encoding** (low cardinality, ≤3 unique values) using `dtype=int` to produce `0`/`1` columns:

| Column | Unique Values |
|---|---|
| `status_code` | 3 (`E`, `P`, `T`) |
| `owner_type_code` | 5 |
| `access_code` | 2 (`public`, `private`) |

**Label Encoding** (higher cardinality):

| Column | Unique Values | Encoded As |
|---|---|---|
| `province` | 11 | `province_encoded` (0–10) |
| `ev_network` | 15 | `ev_network_encoded` (0–14) |

---

## Step 7 — Output Split

The cleaned data was separated into two purpose-built files:

### `ev_chargers_geo.csv` — For Map Visualisation
Contains human-readable fields suitable for plotting on a Canadian map:

- Station name, address, city, province
- Latitude and longitude
- Connector types, network, access hours
- Status code, charger counts

### `ev_chargers_ml.csv` — For Machine Learning (Unscaled)
Contains only numeric and encoded features — **19 columns, 1,719 rows, 0 missing values.**

---

## Step 8 — Feature Scaling

A third file, `ev_chargers_ml_scaled.csv`, was saved with **StandardScaler** applied to continuous columns:

- `latitude`, `longitude`, `ev_dc_fast_count`, `ev_level2_evse_num`

Binary and label-encoded columns were left unscaled. This version is intended for distance-sensitive models such as **K-Nearest Neighbours (KNN)** or **Support Vector Machines (SVM)**.

---

## Final Output Summary

| File | Purpose | Rows | Columns |
|---|---|---|---|
| `ev_chargers_geo.csv` | Map visualisation | 1,719 | 14 |
| `ev_chargers_ml.csv` | ML modelling (unscaled) | 1,719 | 19 |
| `ev_chargers_ml_scaled.csv` | ML modelling (scaled) | 1,719 | 19 |

In [ ]:
"""
EV Charging Station Dataset: Wrangling
1. Clean data for geographic distribution analysis (map plotting)
2. Prepare features for ML (predict future demand / installation locations)

Dataset: Canadian EV charging stations (1,719 rows, all fuel type = ELEC, country = CA)
Source: Kris Strong - Kaggle, https://www.kaggle.com/code/krisstrong/ev-charging-stations-in-canada-10-01-2022/notebook
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

df = pd.read_excel("ev_charging_stations.xlsx")
print(f"Raw shape: {df.shape}")

# 1. Drop columns
COLS_TO_DROP = [
    "Fuel Type Code",           
    "Country",                  
    "Plus4",                    
    "Hydrogen Status Link",     
    "EV Other Info",            
    "BD Blends (French)",       
    "EV On-Site Renewable Source",  
    "Intersection Directions",
    "Access Days Time (French)",
    "Groups With Access Code (French)",
    "EV Pricing (French)",
]
df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], inplace=True)
print(f"After dropping columns: {df.shape}")

# 2. Rename columns using snake_case
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[\s/()]+", "_", regex=True)
      .str.replace(r"[^a-z0-9_]", "", regex=True)
)
print("Columns after rename:\n", df.columns.tolist())


# 3. Fixing data types
# Dates -> datetime
date_cols = ["open_date", "date_last_confirmed", "expected_date", "updated_at"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=False)
df["open_date"] = df["open_date"].dt.strftime("%Y-%m-%d")    # format 


# Numerics -> numeric
num_cols = ["latitude", "longitude", "ev_dc_fast_count", "ev_level2_evse_num", "id"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Boolean -> 0 (false) and 1 (true)
if "restricted_access" in df.columns:
    df["restricted_access"] = df["restricted_access"].map(
        {True: 1, False: 0, "true": 1, "false": 0, 1: 1, 0: 0}
    )

# 4. Handle missing values
if "ev_level2_evse_num" in df.columns:
    df["ev_level2_evse_num"].fillna(0, inplace=True)

# 4c.  categorical columns with mising values are filled with "Unknown"
cat_fill_unknown = [
    "station_phone", "owner_type_code", "facility_type",
    "ev_pricing", "access_detail_code", "ev_network_web",
    "access_days_time",
]
for col in cat_fill_unknown:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# 4d. restricted_access – fill with 0 (unrestricted)
if "restricted_access" in df.columns:
    df["restricted_access"].fillna(0, inplace=True)

# 4e. ev_connector_types – fill with "Unknown"
if "ev_connector_types" in df.columns:
    df["ev_connector_types"].fillna("Unknown", inplace=True)

# 5. Cleaning of geographic data colunms for mapping

CANADA_LAT = (41.0, 84.0)
CANADA_LON = (-141.0, -52.0)

before = len(df)
df = df.dropna(subset=["latitude", "longitude"])   
df = df[
    df["latitude"].between(*CANADA_LAT) &                               # Remove rows with missing or implausible coordinates
    df["longitude"].between(*CANADA_LON)
]
print(f"Rows removed due to invalid coordinates: {before - len(df)}")

# Province name mapping: standardise province codes (abreviated) with province names
PROVINCE_MAP = {
    "AB": "Alberta",        "BC": "British Columbia",  "MB": "Manitoba",
    "NB": "New Brunswick",  "NL": "Newfoundland",       "NS": "Nova Scotia",
    "NT": "Northwest Territories", "NU": "Nunavut",    "ON": "Ontario",
    "PE": "Prince Edward Island",  "QC": "Quebec",     "SK": "Saskatchewan",
    "YT": "Yukon",
}
state_col = next((c for c in df.columns if c in ("state", "st")), None)
if state_col:
    df["province"] = df[state_col].map(PROVINCE_MAP).fillna(df[state_col])


# 6. Encoding categorical data for machine learning

# Low-cardinality: one-hot encoding
LOW_CARD_COLS = ["status_code", "owner_type_code", "access_code"]
df = pd.get_dummies(df, columns=[c for c in LOW_CARD_COLS if c in df.columns],
                    prefix=[c for c in LOW_CARD_COLS if c in df.columns],
                    drop_first=False, dtype=int)

# Province (11 values) -> label encode for tree models
if "province" in df.columns:
    le = LabelEncoder()
    df["province_encoded"] = le.fit_transform(df["province"].astype(str))
    province_classes = dict(enumerate(le.classes_))   # save mapping

# EV Network (15 values) -> label encode
if "ev_network" in df.columns:
    le2 = LabelEncoder()
    df["ev_network_encoded"] = le2.fit_transform(df["ev_network"].astype(str))


# 7. Split into two dataframes and save csv files
#7a. GEO DataFrame  (for mapping / spatial analysis)
GEO_COLS = [
    "id", "station_name", "street_address", "city", "province",
    "latitude", "longitude",
    "status_code" if "status_code" in df.columns else None,
    "ev_network"  if "ev_network"  in df.columns else None,
    "ev_dc_fast_count", "ev_level2_evse_num",
    "ev_connector_types" if "ev_connector_types" in df.columns else None,
    "groups_with_access_code" if "groups_with_access_code" in df.columns else None,
    "access_days_time", "open_date",
]
GEO_COLS = [c for c in GEO_COLS if c and c in df.columns]
df_geo = df[GEO_COLS].copy()
print(f"\nGeo DataFrame shape: {df_geo.shape}")

# 7b. ML DataFrame  (numerical/encoded features only)
# status_code, owner_type_code and access_code: one-hot encoding (0/1)
# province, ec_network: label encoded (single integer column)

DROP_FOR_ML = [
    "station_name", "street_address", "city", "state",
    "zip", "station_phone", "ev_pricing",
    "access_days_time", "access_detail_code", "ev_network_web",
    "groups_with_access_code", "ev_connector_types", "ev_network",
    "geocode_status", "updated_at", "date_last_confirmed",
    "open_date", "expected_date", "facility_type",
    "province",
]
df_ml = df.drop(columns=[c for c in DROP_FOR_ML if c in df.columns])

non_numeric = df_ml.select_dtypes(exclude=[np.number]).columns.tolist()  # Check if all numeric columns is numeric
if non_numeric:
    print(f" Still non-numeric (will label-encode): {non_numeric}")
    for col in non_numeric:
        le_temp = LabelEncoder()
        df_ml[col] = le_temp.fit_transform(df_ml[col].astype(str))

imputer = SimpleImputer(strategy="median")   # Final imputation for any remaining NaNss
df_ml_imputed = pd.DataFrame(
    imputer.fit_transform(df_ml.select_dtypes(include=[np.number])),
    columns=df_ml.select_dtypes(include=[np.number]).columns,
    index=df_ml.index
)
print(f"\nML DataFrame shape: {df_ml_imputed.shape}")
print(f"Remaining NaNs: {df_ml_imputed.isna().sum().sum()}")

# 8. Feature Scaling using StandardScaler()
scaler = StandardScaler()
continuous_features = [                                             # Only scale continuous features; leave binary/encoded as-is
    "latitude", "longitude", "ev_dc_fast_count",
    "ev_level2_evse_num",
]
continuous_features = [c for c in continuous_features if c in df_ml_imputed.columns]
df_ml_scaled = df_ml_imputed.copy()
df_ml_scaled[continuous_features] = scaler.fit_transform(df_ml_imputed[continuous_features])

# 9. Save outputs
df_geo.to_csv("ev_chargers_geo.csv",index=False)
df_ml_imputed.to_csv("ev_chargers_ml.csv",index=False)
df_ml_scaled.to_csv("ev_chargers_ml_scaled.csv",index=False)

print("\nFiles Saved:")
print("- ev_chargers_geo.csv")
print("- ev_chargers_ml.csv")
print("- ev_chargers_ml_scaled.csv")

print("\n── GEO sample ──")
print(df_geo[["station_name", "city", "province", "latitude", "longitude",
              "ev_dc_fast_count"]].head())

print("\n── ML feature summary ──")
print(df_ml_imputed.describe().T[["mean", "std", "min", "max"]].round(2))

Raw shape: (1719, 39)
After dropping columns: (1719, 28)
Columns after rename:
 ['station_name', 'street_address', 'city', 'state', 'zip', 'station_phone', 'status_code', 'expected_date', 'groups_with_access_code', 'access_days_time', 'ev_level2_evse_num', 'ev_dc_fast_count', 'ev_network', 'ev_network_web', 'geocode_status', 'latitude', 'longitude', 'date_last_confirmed', 'id', 'updated_at', 'owner_type_code', 'open_date', 'ev_connector_types', 'access_code', 'access_detail_code', 'facility_type', 'ev_pricing', 'restricted_access']
Rows removed due to invalid coordinates: 0

Geo DataFrame shape: (1719, 14)

ML DataFrame shape: (1719, 19)
Remaining NaNs: 0

Files Saved:
- ev_chargers_geo.csv
- ev_chargers_ml.csv
- ev_chargers_ml_scaled.csv

── GEO sample ──
                                  station_name        city province  \
0                       Essex - Sports Complex       Essex  Ontario   
1  BVD Petroleum - Comber - Tesla Supercharger      Comber  Ontario   
2                   

In [27]:
# Checking data frames
ev_data = pd.read_excel('ev_charging_stations.xlsx')   # original data frame
ev_data.head()

,Fuel Type Code,Station Name,Street Address,Intersection Directions,City,State,ZIP,Plus4,Station Phone,Status Code,...,Access Days Time (French),BD Blends (French),Groups With Access Code (French),Access Code,Access Detail Code,Facility Type,EV Pricing,EV Pricing (French),EV On-Site Renewable Source,Restricted Access
0,ELEC,Essex - Sports Complex,60 Fairview Ave W.,NaN,Essex,ON,N8M 1Y1,NaN,888-356-8911,E,...,Accessible 24 heures par jour,NaN,Public,public,NaN,NaN,NaN,NaN,NaN,NaN
1,ELEC,BVD Petroleum - Comber - Tesla Supercharger,7018 Industrial Drive,NaN,Comber,ON,N0P 1J0,NaN,877-798-3752,E,...,Accessible 24 heures par jour; à l'usage des m...,NaN,Public,public,NaN,GAS_STATION,$0.44 per minute above 60 kW and $0.22 per min...,$0.44 par minute au-dessus de 60 kW et $0.22 p...,NaN,NaN
2,ELEC,Reaume Chevrolet Buick GMC,500 Front Rd,Two units outside; one unit inside,Windsor,ON,N9J 1Z9,NaN,519-734-7844,E,...,24 heures par jour,NaN,Public,public,NaN,CAR_DEALER,Free,Gratuit,NaN,0.0
3,ELEC,Hydro One Inc,5001 Concession Rd 8,NaN,Old Castle,ON,N0R 1L0,NaN,519-737-2634,P,...,NaN,NaN,PRÉVU - pas encore accessible (Public),public,NaN,UTILITY,NaN,NaN,NaN,0.0
4,ELEC,Shell Canada Ltd,5501 Ojibway Pky,NaN,Windsor,ON,N9C 4J5,NaN,NaN,P,...,NaN,NaN,PRÉVU - pas encore accessible (Public),public,NaN,NaN,NaN,NaN,NaN,0.0


In [30]:
# Cleaned dataset (geo - Geographic distribution / mapping dataframe)
ev_geo = pd.read_csv('ev_chargers_geo.csv')
ev_geo.head()

,id,station_name,street_address,city,province,latitude,longitude,ev_network,ev_dc_fast_count,ev_level2_evse_num,ev_connector_types,groups_with_access_code,access_days_time,open_date
0,129454,Essex - Sports Complex,60 Fairview Ave W.,Essex,Ontario,42.167112,-82.816715,FLO,2,0.0,CHADEMO J1772COMBO,Public,24 hours daily,2019-07-17
1,83915,BVD Petroleum - Comber - Tesla Supercharger,7018 Industrial Drive,Comber,Ontario,42.239116,-82.550227,Tesla,8,0.0,TESLA,Public,24 hours daily; for Tesla use only,2015-04-01
2,83263,Reaume Chevrolet Buick GMC,500 Front Rd,Windsor,Ontario,42.241722,-83.102857,Non-Networked,1,1.0,J1772 J1772COMBO,Public,24 hours daily,2013-01-15
3,156672,Hydro One Inc,5001 Concession Rd 8,Old Castle,Ontario,42.243330,-82.947170,Non-Networked,2,0.0,CHADEMO J1772COMBO,PLANNED - not yet accessible (Public),Unknown,NaN
4,210228,Shell Canada Ltd,5501 Ojibway Pky,Windsor,Ontario,42.260096,-83.084966,Non-Networked,2,0.0,Unknown,PLANNED - not yet accessible (Public),Unknown,NaN


In [29]:
# ML dataframe (scaled data)
ev_scaled = pd.read_csv('ev_chargers_ml_scaled.csv')
ev_scaled.head()

,ev_level2_evse_num,ev_dc_fast_count,latitude,longitude,id,restricted_access,status_code_E,status_code_P,status_code_T,owner_type_code_FG,owner_type_code_LG,owner_type_code_P,owner_type_code_SG,owner_type_code_T,owner_type_code_Unknown,access_code_private,access_code_public,province_encoded,ev_network_encoded
0,-0.327433,-0.184706,-1.543743,0.246085,129454.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,6.0,7.0
1,-0.327433,1.890516,-1.521982,0.258588,83915.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,13.0
2,0.807359,-0.530576,-1.521195,0.232661,83263.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,9.0
3,-0.327433,-0.184706,-1.520709,0.239965,156672.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,9.0
4,-0.327433,-0.184706,-1.515642,0.233500,210228.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,9.0
